**Purpose:** LLM labeling of news snippets (Qwen / Gemini / Llama batches in this folder).

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.secrets import HF_TOKEN  # copy src/secrets_example.py to src/secrets.py


In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"                     # [315e0, 69797]
#model_name = "google/gemma-7b-it"                           # [315e0]
#model_name = "meta-llama/Llama-3.1-8B-Instruct"             # [315e0, 9d5c9, 69797]

#!pip install --upgrade pip
#!pip install --upgrade --no-deps requests pandas
#!pip install --upgrade --no-deps spacy==3.8.7
#!pip install --upgrade numpy==1.26.4
#!python -m spacy download en_core_web_sm
#!pip install -q transformers accelerate torch huggingface_hub pandas

!pip install --upgrade pip
!pip install --upgrade --no-deps requests pandas numpy torch transformers==4.35.0 accelerate huggingface_hub==0.17.4 spacy==3.8.7 pyarrow
!python -m spacy download en_core_web_sm


from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import StoppingCriteria, StoppingCriteriaList
from huggingface_hub import login
import torch
import time
import re
import random
import json
import pandas as pd
import hashlib
import requests, zipfile, io
import spacy
nlp = spacy.load("en_core_web_sm")
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def deterministic_shuffle(lista_datas):
    def digital_root(n: int) -> int:
        while n > 9:
            s = 0
            while n > 0:
                s += n % 10
                n //= 10
            n = s
        return n
    dates = lista_datas
    encoded = {
        int(hashlib.md5(x.encode()).hexdigest(), 16): x
        for x in dates
    }    
    encoded_sorted = dict(sorted(encoded.items()))
    encoded_split = {k: (digital_root(k), v) for k,v in encoded_sorted.items()}
    return encoded_split

def clean_quotations(text):
    text = re.sub(r'\d+\|\d+\|\|', '', text)
    text = re.sub(r'#\d+', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s([.,!?;:])', r'\1', text)
    return text.strip()
    
def lemmatize_quotation(text):
    doc = nlp(text)
    lemmas = [token.lemma_ for token in doc]
    return " ".join(lemmas)

def load_gdelt_news(timestamp):
    url = f"http://data.gdeltproject.org/gdeltv2/{timestamp}.gkg.csv.zip"
    r = requests.get(url)
    try:
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            with z.open(z.namelist()[0]) as f:
                df = pd.read_csv(f, sep='\t', header=None, dtype=str, encoding='ISO-8859-1')
                df.columns = [
                    "GKGRECORDID", "DATE", "SourceCollectionIdentifier", "SourceCommonName",
                    "DocumentIdentifier", "Counts", "V2Counts", "Themes", "V2Themes",
                    "Locations", "V2Locations", "Persons", "V2Persons", "Organizations",
                    "V2Organizations", "V2Tone", "Dates", "GCAM", "SharingImage",
                    "RelatedImages", "SocialImageEmbeds", "SocialVideoEmbeds", "Quotations",
                    "AllNames", "Amounts", "TranslationInfo", "Extras"
                ]
                df = df[["GKGRECORDID", "DATE", "SourceCommonName", "Quotations"]]
                df = df[df["SourceCommonName"].isin(sources)]
                df = df[df["Quotations"].notna()]
                df["Quotations"] = df["Quotations"].map(clean_quotations)
                df = df.drop_duplicates(subset=["SourceCommonName", "Quotations"])
                df["search_Quotations"] = df["Quotations"].map(lemmatize_quotation)
                sector_dfs = {}
                for sector, pattern in sector_patterns.items():
                    sector_dfs[sector] = df[df["SourceCommonName"].isin(news_settings["SourceCommonName_filter"][sector])]
                    sector_dfs[sector] = sector_dfs[sector][sector_dfs[sector]["search_Quotations"].str.contains(pattern, na=False, flags=re.IGNORECASE)]
    except Exception as e:
        error_message = f"Failed to process {url}: {e}\n"
        with open("errors.log", "a") as log_file:
            log_file.write(error_message)
        return False
    return sector_dfs

In [ ]:
class StopAtJSONEnd(StoppingCriteria):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.end = "}"
        self.buffer = ""

    def __call__(self, input_ids, scores, **kwargs):
        text = self.tokenizer.decode(input_ids[0], skip_special_tokens=True)
        if text.strip().endswith("}"):
            return True
        return False

login(token=HF_TOKEN)  # from src.secrets (see src/secrets_example.py)

# --- Load model and tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model on GPU automatically
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# --- Define prompt ---
def label_word_sector(news_quote, news_sector, news_id, model, tokenizer, max_new_tokens=500):
    prompt = """
You are a financial-domain impact classifier for equity sectors. 
Your task is to analyze a GDELT news quote and classify the expected impact of the described events on a specific GICS sector.

Classification Rules:
- "positive": the event is expected to benefit the sector (e.g., increased demand, revenue growth, cost reduction, regulatory advantage, market share gain).
- "negative": the event is expected to harm the sector (e.g., decreased demand, revenue loss, higher costs, regulatory pressure, operational disruption).
- "neutral": the event has no meaningful impact, is unrelated, or the impact is ambiguous/uncertain.

Definition of "meaningful impact":
- There must be an explicit or implied causal link between the event and sector performance.
- The event should clearly suggest an effect on revenue, costs, operations, regulatory conditions, or valuation.
- Interpretation is allowed, but it must be logically justified with cause-effect reasoning.

Non-meaningful impact:
- Mere mentions of companies, sectors, or locations without linkage to sector performance.
- Facts, statistics, or descriptions without explanation of consequences.
- Speculative statements with no plausible sector-level effect.

Output Requirements:
Return a single JSON object **exactly** in the following format:

{
  "sector": "<sector name>",
  "sentiment": "<positive | negative | neutral>",
  "rationale": "<3–5 sentence explanation linking the event to sector impact>",
  "confidence": <float between 0 and 1>
}

- "rationale": 2–5 sentence explanation of why the impact label was chosen, highlighting the causal link.
- "confidence": float between 0.0 (very unsure) to 1.0 (very confident) representing your certainty.
- Default label is "neutral" if impact is unclear or not meaningful.

Content to classify:

Sector: """ + news_sector + """
Quote: """ + news_quote + """

--- END OF CONTENT ---

Classify strictly according to these rules. Output only the JSON object, nothing else."""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    stop_criteria = StoppingCriteriaList([StopAtJSONEnd(tokenizer)])
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.0,
            stopping_criteria=stop_criteria
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    text_generated = text.split("--- END OF CONTENT ---")[-1].strip()
    text_generated = text_generated.strip().replace("{{", "{").replace("}}", "}")
    match = re.search(r"\{.*?\}", text_generated, re.S)

    print(text_generated)
    print()
    print()
    
    if not match:
        with open("errors.log", "a") as f:
            f.write(text + "\n")
            f.write(str(news_id) + " " + "no JSON" + "\n")
        return {"news_id": news_id, "error": "no JSON"}
    try:
        return json.loads(match.group(0))
    except Exception:
        with open("errors.log", "a") as f:
            f.write(text + "\n")
            f.write(str(news_id) + " " + "invalid JSON" + "\n")
        return {"news_id": news_id, "error": "invalid JSON"}

In [ ]:
# LOAD EVERYTHNG INTO MEMORY

with open("/kaggle/input/news-json/news_0.json", "r") as f:
    news_settings = json.load(f)

sector_patterns = {
    sector: r"(?<!\w)(?:" + "|".join(map(re.escape, words)) + r")(?!\w)"
    for sector, words in news_settings["Quotations_filter"].items()
}

sources = {source for lst in news_settings["SourceCommonName_filter"].values() for source in lst}

timestamps = deterministic_shuffle(news_settings["DATE_filter"])


# READY TO START ?

last_hash = 1000
bucket_to_do = 1

max_duration = 10 * 3600
start_time = time.time()

timestamps_count = 0
records_count = 0
records = []

for current_hash, (bucket, timestamp) in timestamps.items():
    elapsed_time = time.time() - start_time
    if elapsed_time > max_duration:
        print(f"⏰ Time limit reached ({max_duration / 3600} hours). Stopping the loop.")
        break

    if last_hash >= current_hash:
        continue

    if bucket == bucket_to_do:
        pass
    else:
        continue
        
    news_dfs = load_gdelt_news(timestamp)
    if not news_dfs:
        continue
    
    print(f"Running {timestamp} ({current_hash})")
    timestamps_count += 1
    
    for sector, df in news_dfs.items():
        for i, row in df.iterrows():
            records_count += 1
            gdeltid = row["GKGRECORDID"]
            sector = sector
            quote = row["Quotations"]

            try:
                result = label_word_sector(quote, sector, gdeltid, model, tokenizer)
                torch.cuda.empty_cache()
        
                records.append({
                    "GKGRECORDID": f"{gdeltid} ({sector})",
                    f"{model_name}": result
                })
    
                if random.random() < 0.025:
                    with open("validate.txt", "a", encoding="utf-8") as f:
                        f.write(str(sector) + "\n\n")
                        f.write(str(quote) + "\n\n")
                        f.write(str(result) + "\n")
                        f.write("\n" * 4)
        
            except RuntimeError as e:
                with open("failed.txt", "w") as f:
                    f.write(str(gdeltid) + " : " + str(e) + "\n")
                continue

df = pd.DataFrame(records)
df.to_parquet(f"news_{model_name.split('/')[0]}_{str(bucket_to_do)}_{str(current_hash)}.parquet", engine="pyarrow", index=False)

with open("last_hash.txt", "w") as f:
    f.write(str(current_hash))
    f.write("\n Timestamps count: " + str(timestamps_count))
    f.write("\n Records count: " + str(records_count))
print("END")